In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json
from tqdm import tqdm
import torchvision.models as torch_models
from torchvision import transforms
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import PreTrainedTokenizerFast

# ============ 1. Create tokenizer ============
print("=" * 60)
print("TOKENIZER SETUP")
print("=" * 60)

tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print(f"Vocabulary: {tokenizer.get_vocab()}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}\n")

# ============ 2. FIXED ResNet + GPT2 Model ============
class ResNetGPT2PrefixModel(nn.Module):
    """
    ResNet encoder + GPT2 decoder with PREFIX-TUNING
    """
    def __init__(self, vocab_size, gpt2_hidden_size=128, num_prefix_tokens=16):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = gpt2_hidden_size
        self.num_prefix_tokens = num_prefix_tokens
        
        # ResNet18 encoder
        resnet = torch_models.resnet18(pretrained=True)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-1])
        
        # Project ResNet features to GPT2 dimension
        self.feature_projection = nn.Linear(512, gpt2_hidden_size)
        
        # Generate prefix embeddings from image
        self.prefix_generator = nn.Sequential(
            nn.Linear(gpt2_hidden_size, gpt2_hidden_size * 2),
            nn.Tanh(),
            nn.Linear(gpt2_hidden_size * 2, num_prefix_tokens * gpt2_hidden_size)
        )
        
        # GPT2 decoder
        gpt2_config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=128,
            n_embd=gpt2_hidden_size,
            n_layer=2,
            n_head=2,
            resid_pdrop=0.4,
            embd_pdrop=0.4,
            attn_pdrop=0.4,
        )
        self.gpt2 = GPT2LMHeadModel(gpt2_config)
        
    def encode_image_to_prefix(self, images):
        """Convert images to prefix embeddings"""
        batch_size = images.shape[0]
        features = self.resnet_features(images).squeeze(-1).squeeze(-1)
        features = self.feature_projection(features)
        prefix_flat = self.prefix_generator(features)
        prefix_embeds = prefix_flat.view(batch_size, self.num_prefix_tokens, self.hidden_size)
        return prefix_embeds
    
    def forward(self, images, input_ids, labels=None):
        """
        FIXED: Proper label handling for prefix-tuning
        """
        prefix_embeds = self.encode_image_to_prefix(images)
        token_embeds = self.gpt2.transformer.wte(input_ids)
        inputs_embeds = torch.cat([prefix_embeds, token_embeds], dim=1)
        
        # CRITICAL FIX: If labels provided, they should match the OUTPUT positions
        # Output has: [prefix_logits (ignored), token_logits]
        # We only compute loss on token_logits, which correspond to our labels
        if labels is not None:
            # The model outputs logits for: [prefix positions] + [token positions]
            # We want to compute loss only on token positions
            # Shift labels to align with output positions AFTER prefix
            outputs = self.gpt2(inputs_embeds=inputs_embeds, return_dict=True)
            
            # Extract logits for non-prefix positions
            # outputs.logits shape: [batch, prefix_len + seq_len, vocab]
            # We want logits starting from position num_prefix_tokens
            logits = outputs.logits[:, self.num_prefix_tokens:, :]  # [batch, seq_len, vocab]
            
            # Compute loss manually
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
            
            outputs.loss = loss
            return outputs
        else:
            return self.gpt2(inputs_embeds=inputs_embeds, return_dict=True)
    
    def generate(self, images, max_length=25, bos_token_id=1, eos_token_id=2, pad_token_id=0):
        """
        FIXED: Proper generation with prefix
        """
        self.eval()
        batch_size = images.shape[0]
        device = images.device
        
        prefix_embeds = self.encode_image_to_prefix(images)
        generated_ids = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        
        for step in range(max_length - 1):
            token_embeds = self.gpt2.transformer.wte(generated_ids)
            inputs_embeds = torch.cat([prefix_embeds, token_embeds], dim=1)
            
            outputs = self.gpt2(inputs_embeds=inputs_embeds, return_dict=True)
            
            # CRITICAL FIX: Get logits for the last TOKEN position (not last overall position)
            # The last position in outputs.logits corresponds to the last token we fed in
            next_token_logits = outputs.logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1)
            
            # Mark finished sequences
            finished = finished | (next_token == eos_token_id)
            
            # Replace tokens in finished sequences with pad
            next_token = torch.where(finished, torch.tensor(pad_token_id, device=device), next_token)
            
            generated_ids = torch.cat([generated_ids, next_token.unsqueeze(1)], dim=1)
            
            # Stop if all sequences are finished
            if finished.all():
                break
        
        return generated_ids

# ============ 3. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, transform, tokenizer):
        self.entries = entries
        self.transform = transform
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.entries)
    
    def __getitem__(self, idx):
        entry = self.entries[idx]
        img = Image.open(entry["image"]).convert("RGB")
        img_tensor = self.transform(img)
        
        # CRITICAL FIX: Manually add special tokens since tokenizer isn't doing it
        sequence_text = " ".join(entry["sequence"])
        tokens = self.tokenizer.encode(sequence_text, add_special_tokens=False)
        
        # Manually add BOS and EOS
        input_ids = torch.tensor(
            [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id],
            dtype=torch.long
        )
        
        return img_tensor, input_ids, entry["sequence"]

def collate_fn(batch, pad_token_id=0):
    images = torch.stack([item[0] for item in batch])
    sequences = [item[1] for item in batch]
    originals = [item[2] for item in batch]
    
    max_len = max(len(s) for s in sequences)
    padded = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    
    return images, padded, originals

# ============ 4. FIXED Training ============
def train_model(model, train_loader, epochs, lr, device):
    model = model.to(device)
    
    optimizer = torch.optim.AdamW([
        {'params': model.resnet_features.parameters(), 'lr': lr / 10},
        {'params': model.feature_projection.parameters(), 'lr': lr},
        {'params': model.prefix_generator.parameters(), 'lr': lr},
        {'params': model.gpt2.parameters(), 'lr': lr / 5},
    ], weight_decay=0.01)
    
    # Add learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, sequences, _ in progress_bar:
            images = images.to(device)
            sequences = sequences.to(device)
            
            # FIXED: Proper input/label preparation
            # Input: <s> R R R ...
            # Label: R R R ... </s>
            input_ids = sequences[:, :-1]  # Everything except last token
            labels = sequences[:, 1:].clone()  # Everything except first token
            
            # Mask padding tokens in labels
            labels[labels == 0] = -100
            
            # Forward pass - model now handles prefix alignment internally
            outputs = model(images, input_ids, labels)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        scheduler.step()  # Update learning rate
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.2e}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
        
        # Test every 25 epochs
        if (epoch + 1) % 25 == 0:
            test_model(model, train_loader, device, tokenizer, num_samples=5)
            model.train()
    
    return best_loss

def test_model(model, loader, device, tokenizer, num_samples=10):
    model.eval()
    dataset = loader.dataset
    
    # Only print detailed predictions for first 20
    print_detailed = num_samples <= 20
    
    if print_detailed:
        print("\n" + "="*60)
        print("Sample Predictions:")
        print("="*60)
    else:
        print(f"\nEvaluating on {num_samples} mazes...")
    
    correct = 0
    
    # Use tqdm for progress bar when testing many samples
    iterator = range(min(num_samples, len(dataset)))
    if num_samples > 20:
        iterator = tqdm(iterator, desc="Testing")
    
    for i in iterator:
        img_tensor, _, original_seq = dataset[i]
        img_tensor = img_tensor.unsqueeze(0).to(device)
        
        with torch.no_grad():
            generated = model.generate(
                img_tensor,
                max_length=25,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        
        predicted = tokenizer.decode(generated[0], skip_special_tokens=True)
        expected = " ".join(original_seq)
        
        match = predicted == expected
        if match:
            correct += 1
        
        # Only print details for first 20
        if print_detailed:
            print(f"\nMaze {i}:")
            print(f"  Expected:  '{expected}'")
            print(f"  Predicted: '{predicted}'")
            print(f"  Match: {'✓' if match else '✗'}")
    
    print(f"\nAccuracy: {correct}/{num_samples} ({100*correct/num_samples:.1f}%)\n")
    return correct

# ============ 5. Main ============
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train_entries = json.load(open("data/train_sequences.json"))
test_entries = json.load(open("data/test_sequences.json"))

print(f"Training set: {len(train_entries)} examples")
print(f"Test set: {len(test_entries)} examples")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = MazeDataset(train_entries, transform, tokenizer)
test_dataset = MazeDataset(test_entries, transform, tokenizer)

# VERIFY SPECIAL TOKENS ARE NOW INCLUDED
print("=" * 60)
print("VERIFICATION - Special Tokens")
print("=" * 60)
sample_img, sample_ids, sample_seq = dataset[0]
print(f"Sample sequence: {sample_seq}")
print(f"Token IDs: {sample_ids.tolist()}")
print(f"BOS at start? {sample_ids[0].item() == tokenizer.bos_token_id}")
print(f"EOS at end? {sample_ids[-1].item() == tokenizer.eos_token_id}")
print(f"Decoded: '{tokenizer.decode(sample_ids)}'")
print(f"Expected format: '<s> R R R ... </s>'\n")

train_loader = DataLoader(
    dataset, 
    batch_size=32,
    shuffle=True, 
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = ResNetGPT2PrefixModel(vocab_size=len(tokenizer), gpt2_hidden_size=64, num_prefix_tokens=8)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Prefix tokens: {model.num_prefix_tokens}\n")

# print("=" * 60)
# print("TRAINING - FIXED ResNet18 + GPT2 with PREFIX-TUNING")
# print("=" * 60)
# print("✓ Fixed label alignment for prefix-tuning")
# print("✓ Fixed generation logic")
# print("✓ Proper loss computation\n")

print("\n" + "=" * 60)
print("FINAL TRAINING SET EVALUATION")
print("=" * 60)
print(f"Evaluating on ALL {len(train_entries)} training mazes...")
print("=" * 60)

final_loss = train_model(model, train_loader, epochs=75, lr=5e-4, device=device)


print("\n" + "=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)
print(f"Testing on UNSEEN test set ({len(test_entries)} mazes)")
print("=" * 60)

# Evaluate on training set
train_correct = test_model(model, train_loader, device, tokenizer, num_samples=len(train_entries))
print("=" * 60)
print(f"Training Accuracy: {train_correct}/{len(train_entries)} ({100*train_correct/len(train_entries):.1f}%)")
print("=" * 60)

print("\n" + "=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)
print(f"Testing on UNSEEN test set ({len(test_entries)} mazes)")
print("=" * 60)

# Test on the held-out test set
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

test_correct = test_model(model, test_loader, device, tokenizer, num_samples=len(test_entries))

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Final Training Loss: {final_loss:.6f}")
print(f"Training Accuracy: {train_correct}/{len(train_entries)} ({100*train_correct/len(train_entries):.1f}%)")
print(f"Test Accuracy: {test_correct}/{len(test_entries)} ({100*test_correct/len(test_entries):.1f}%)")
print(f"Generalization Gap: {100*train_correct/len(train_entries) - 100*test_correct/len(test_entries):.1f}%")
print("=" * 60)

if final_loss < 0.1 and test_correct >= 0.8 * len(test_entries):
    print("\n🎉 SUCCESS! Model generalizes well to unseen mazes!")
elif final_loss < 0.2 and test_correct >= 0.6 * len(test_entries):
    print("\n✓ Good progress! Model is learning but could improve")
else:
    print("\n⚠️ May need more training or hyperparameter tuning")

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
    'num_prefix_tokens': model.num_prefix_tokens,
}, "resnet_gpt2_prefix.pth")
print("\n💾 Model saved to resnet_gpt2_prefix.pth")

TOKENIZER SETUP
Vocabulary: {'U': 5, '<s>': 1, '<pad>': 0, 'R': 4, '</s>': 2, '<unk>': 3}
Vocab size: 6
Special tokens: BOS=1, EOS=2, PAD=0

LOADING DATA
Training set: 36950 examples
Test set: 9250 examples
VERIFICATION - Special Tokens
Sample sequence: ['R', 'U', 'U', 'U', 'R', 'U', 'R', 'R', 'U', 'R', 'R', 'U']
Token IDs: [1, 4, 5, 5, 5, 4, 5, 4, 4, 5, 4, 4, 5, 2]
BOS at start? True
EOS at end? True
Decoded: '<s> R U U U R U R R U R R U </s>'
Expected format: '<s> R R R ... </s>'

Device: cuda
Parameters: 11,952,320
Prefix tokens: 8


FINAL TRAINING SET EVALUATION
Evaluating on ALL 36950 training mazes...


Epoch 1/75: 100%|██████████| 1155/1155 [05:58<00:00,  3.23it/s, loss=0.4869]


Epoch 1, Avg Loss: 0.569403, LR: 5.00e-05


Epoch 2/75: 100%|██████████| 1155/1155 [02:38<00:00,  7.30it/s, loss=0.4305]


Epoch 2, Avg Loss: 0.456391, LR: 4.99e-05


...

Epoch 75, Avg Loss: 0.001987, LR: 1.00e-06

Sample Predictions:

Maze 0:
  Expected:  'R U U U R U R R U R R U'
  Predicted: 'R U U U R U R R U R R U'
  Match: ✓

Maze 1:
  Expected:  'R U U U R U R R U R R U'
  Predicted: 'R U U U R U R R U R R U'
  Match: ✓

Maze 2:
  Expected:  'R U U U R U R R U R R U'
  Predicted: 'R U U U R U R R U R R U'
  Match: ✓

Maze 3:
  Expected:  'R U U U R U R R U R R U'
  Predicted: 'R U U U R U R R U R R U'
  Match: ✓

Maze 4:
  Expected:  'R U U U R U R R U R R U'
  Predicted: 'R U U U R U R R U R R U'
  Match: ✓

Accuracy: 5/5 (100.0%)


FINAL TEST SET EVALUATION
Testing on UNSEEN test set (9250 mazes)

Evaluating on 36950 mazes...


Testing: 100%|██████████| 36950/36950 [26:32<00:00, 23.20it/s]



Accuracy: 36940/36950 (100.0%)

Training Accuracy: 36940/36950 (100.0%)

FINAL TEST SET EVALUATION
Testing on UNSEEN test set (9250 mazes)

Evaluating on 9250 mazes...


Testing: 100%|██████████| 9250/9250 [08:00<00:00, 19.25it/s]


Accuracy: 2854/9250 (30.9%)

FINAL RESULTS
Final Training Loss: 0.001987
Training Accuracy: 36940/36950 (100.0%)
Test Accuracy: 2854/9250 (30.9%)
Generalization Gap: 69.1%

⚠️ May need more training or hyperparameter tuning

💾 Model saved to resnet_gpt2_prefix.pth
